In [49]:
import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt
from sklearn.metrics import root_mean_squared_error
import statsmodels.api as sm
from statsmodels.regression.linear_model import OLS
import math
from sklearn.model_selection import train_test_split

### Forward selection

In [138]:
numrows = 1000
numcols = 10
np.random.seed(0)
X = np.random.normal(0, 1, (numrows, numcols))
coefs = np.random.uniform(1, 5, numcols)
Y = X @ coefs + np.random.normal(0, 1, numrows)

In [ ]:
[3, 6, 7, 2, 5, 1, 9, 4, 0, 8]

In [139]:
coefs

array([1.38266659, 2.60768308, 3.10220095, 3.9293287 , 1.5411149 ,
       2.76092265, 3.78260759, 3.53155077, 1.3361475 , 2.34574745])

In [140]:
results = sm.OLS(Y, X).fit()
results.params

array([1.42290952, 2.5569784 , 3.05061446, 3.9116331 , 1.56054188,
       2.75686509, 3.78723757, 3.53954894, 1.36396238, 2.36977678])

In [143]:
best_rsquared = -np.inf
best_col = None
for n in range(numcols):
    results = sm.OLS(Y, X[:,n]).fit()
    if results.rsquared > best_rsquared:
        best_rsquared = results.rsquared
        best_col = n
print(best_col, best_rsquared)
best_rsquared2 = -np.inf
best_col2 = None
for n in range(numcols):
    if n == best_col:
        continue
    results = sm.OLS(Y, X[:,[best_col, n]]).fit()
    if results.rsquared > best_rsquared2:
        best_rsquared2 = results.rsquared
        best_col2 = n
print(best_col2, best_rsquared2)

3 0.17522277749183868
6 0.3502590039906145


In [145]:
10 * 11 / 2

55.0

In [161]:
numrows = 1000
numcols = 10
np.random.seed(0)
X = np.random.normal(0, 1, (numrows, numcols))
coefs = np.random.uniform(1, 5, (numcols,))
Y = X @ coefs + np.random.normal(0, 1, (numrows,))

In [162]:
coefs

array([1.38266659, 2.60768308, 3.10220095, 3.9293287 , 1.5411149 ,
       2.76092265, 3.78260759, 3.53155077, 1.3361475 , 2.34574745])

In [163]:
cols_used = list()
rsq_obtained = list()
cols_unused = np.arange(numcols)

In [164]:
cols_used.append(1)

In [165]:
cols_used

[1]

In [166]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.33, random_state=42)

In [167]:
# Init
cols_used = list()
rsq_obtained = list()
cols_unused = np.arange(numcols)

for n in range(numcols):
    cur_col = ""
    cur_rsq = -np.inf
    for col in cols_unused:
        cols_used_extended = list(set(cols_used).union({col}))
        X_next = X_train[:, cols_used_extended]
        results = sm.OLS(y_train, X_next).fit()
        rsq = results.rsquared
        if rsq > cur_rsq:
            cur_col = col
            cur_rsq = rsq
    cols_unused = list(set(cols_unused) - {cur_col})
    cols_used.append(cur_col)
    rsq_obtained.append(cur_rsq)
print(cols_used)
print([f"{x:.3}" for x in rsq_obtained])

[3, 6, 7, 2, 5, 1, 9, 4, 0, 8]
['0.166', '0.339', '0.516', '0.634', '0.726', '0.818', '0.892', '0.926', '0.957', '0.985']


In [160]:
rmse_list = list()
for n in range(numcols):
    cols_list = cols_used[0:n+1]
    results = sm.OLS(y_train, X_train[:, cols_list]).fit()
    rmse_list.append(root_mean_squared_error(y_test, results.predict(X_test[:, cols_list])))
rmse_list

[11.896568112926204,
 10.7819798287256,
 11.47024313568041,
 9.035440999050513,
 8.511452821141946,
 6.433333557228374,
 5.331584796425593,
 3.6859282787368515,
 2.3034995047124536,
 2.4570287688075862]

### Backward selection

In [58]:
cols_unused = list()
rsq_obtained = list()
cols_used = np.arange(numcols)
for n in range(numcols - 1):
    cur_col = ""
    cur_rsq = -np.inf
    for col in cols_used:
        cols_used_reduced = list(set(cols_used) - {col})
        X_next = X[:, cols_used_reduced]
        results = sm.OLS(Y, X_next).fit()
        rsq = results.rsquared
        if rsq > cur_rsq:
            cur_col = col
            cur_rsq = rsq
    cols_used = list(set(cols_used) - {cur_col})
    cols_unused.append(cur_col)
    rsq_obtained.append(cur_rsq)
print(cols_unused)
print([f"{x:.3}" for x in rsq_obtained])

[8, 0, 4, 9, 1, 5, 2, 7, 6]
['0.959', '0.931', '0.9', '0.829', '0.741', '0.655', '0.531', '0.35', '0.175']


### Lasso and Ridge Regression

In [176]:
numrows = 1000
numcols = 10
np.random.seed(0)
X = np.random.normal(0, 1, (numrows, numcols))

In [177]:
coefs = np.hstack((np.zeros(numcols // 2), np.random.uniform(1, 5, (numcols // 2,))))
coefs

array([0.        , 0.        , 0.        , 0.        , 0.        ,
       1.38266659, 2.60768308, 3.10220095, 3.9293287 , 1.5411149 ])

In [178]:
#coefs = np.hstack((np.zeros(990), np.random.uniform(1, 5, (10,))))
#coefs

In [179]:
Y = X @ coefs + np.random.normal(0, 1, (numrows,))

In [180]:
results = sm.OLS(Y, X).fit()
results.params

array([ 2.79121326e-03, -4.87212951e-02, -4.79716317e-02, -2.14565820e-03,
       -2.83773545e-02,  1.36771148e+00,  2.65412379e+00,  3.05627918e+00,
        3.91580564e+00,  1.58230918e+00])

### Lasso

In [189]:
results = OLS(Y, X).fit_regularized(method = 'elastic_net', alpha = 1, L1_wt = 1.0)

In [190]:
results.params

array([0.        , 0.        , 0.        , 0.        , 0.        ,
       0.30769623, 1.54362766, 2.10149979, 2.95571092, 0.4342407 ])

In [191]:
root_mean_squared_error(Y, X @ results.params)

2.5050287180556654

### Null model - mean only

In [123]:
root_mean_squared_error(Y, Y.mean() * np.ones(Y.shape[0])) # null model - just predict the mean

6.099144517146659

### Refit model - assume Lasso's chosen coefs are all that matters

In [192]:
X2 = X[:, np.argwhere(results.params > 0).ravel()]
results2 = OLS(Y, X2).fit()

In [193]:
results2.params

array([1.37081265, 2.65497889, 3.05899617, 3.91627845, 1.583533  ])

In [127]:
results2.rsquared

0.9722406311753975

In [128]:
root_mean_squared_error(Y, X2 @ results2.params)

1.0165357887493351

### Ridge

In [195]:
results3 = OLS(Y, X).fit_regularized(method = 'elastic_net', alpha = 0.1, L1_wt = 0.0)

In [196]:
results3.params

array([ 0.0046117 , -0.05249398, -0.05800463, -0.01746965, -0.0419326 ,
        1.23270908,  2.38793752,  2.78874809,  3.57639994,  1.4174643 ])

In [197]:
root_mean_squared_error(Y, X @ results3.params)

1.152208066368024

### Test data

In [73]:
X_test = np.random.normal(0, 1, (numrows, numcols))
Y_test = X_test @ coefs + np.random.normal(0, 1, (numrows,))

In [74]:
root_mean_squared_error(Y_test, X_test @ results.params)

2.2712726717197023

In [75]:
df_test = pd.DataFrame(X_test, columns = list(range(10)))
X_test_cut = df_test.iloc[:, np.argwhere(coefs > 0).ravel()]
root_mean_squared_error(Y_test, X_test_cut @ results2.params)

1.016817858147878

In [76]:
root_mean_squared_error(Y_test, X_test @ results3.params)

1.1492647139780656